# Maize project: Colab launcher
No logic lives here. Everything is in the repo's `scripts/` and `src/maize/`.
Runtime > Change runtime type > GPU before running.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
REPO_URL = 'https://github.com/<you>/<repo>.git'   # TODO
DRIVE = '/content/drive/MyDrive/maize'              # data zip + runs live here

# fast local copy of data; runs go straight to Drive so they survive disconnects
os.environ['MAIZE_DATA_ROOT'] = '/content/data'
os.environ['MAIZE_RUNS_ROOT'] = f'{DRIVE}/runs'

!rm -rf /content/repo && git clone $REPO_URL /content/repo
%cd /content/repo
# TensorFlow/OpenCV already exist on Colab: do not reinstall them
!pip install -q --no-deps -e .
!pip install -q imagehash
!python -c "import tensorflow as tf; print('TF', tf.__version__, tf.config.list_physical_devices('GPU'))"

In [ ]:
# first session only: unpack the data (raw images + manifest + processed cache) from Drive
!mkdir -p /content/data && unzip -q -n {DRIVE}/data.zip -d /content/data
!ls /content/data | head

In [ ]:
# one run per cell; finished folds are skipped, so re-running after a disconnect resumes
!python scripts/05_train_classifier.py --config configs/classifier/mobilenetv2.yaml --fold all

In [ ]:
!python scripts/06_train_ae.py --config configs/autoencoder/cae_ssim.yaml --fold 0